In [ ]:
%%html
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');

body, .jp-Notebook { background: #f0f2f5 !important; font-family: 'Inter', sans-serif !important; }

.hero {
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%);
    padding: 2.5rem 2rem;
    border-radius: 16px;
    margin-bottom: 1.5rem;
    color: white;
}
.hero h1 { font-size: 2rem; font-weight: 700; margin: 0 0 0.4rem 0; letter-spacing: -0.5px; }
.hero p  { font-size: 0.95rem; opacity: 0.75; margin: 0; font-weight: 300; }
.hero-badge {
    display: inline-block;
    background: rgba(255,255,255,0.15);
    border: 1px solid rgba(255,255,255,0.25);
    border-radius: 20px;
    padding: 3px 14px;
    font-size: 0.75rem;
    font-weight: 500;
    margin-bottom: 0.8rem;
    letter-spacing: 0.5px;
}

.kpi-row { display: flex; gap: 1rem; margin-bottom: 1.5rem; flex-wrap: wrap; }
.kpi-card {
    flex: 1; min-width: 180px;
    background: white;
    border: 1px solid #e8ecf0;
    border-radius: 12px;
    padding: 1.2rem 1.4rem;
    box-shadow: 0 1px 6px rgba(0,0,0,0.06);
    position: relative;
}
.kpi-label { font-size: 0.7rem; font-weight: 600; text-transform: uppercase; letter-spacing: 0.8px; color: #8492a6; margin-bottom: 0.3rem; }
.kpi-value { font-size: 1.8rem; font-weight: 700; color: #1a1a2e; line-height: 1; margin-bottom: 0.2rem; }
.kpi-sub   { font-size: 0.75rem; color: #8492a6; }
.kpi-icon  { font-size: 2rem; position: absolute; top: 1rem; right: 1.2rem; opacity: 0.12; }

.section-header { font-size: 1rem; font-weight: 600; color: #1a1a2e; margin: 0.5rem 0 0.2rem 0; }
.section-desc   { font-size: 0.82rem; color: #8492a6; margin-bottom: 0.8rem; }

.fancy-divider { border: none; height: 1px; background: linear-gradient(to right, transparent, #d8dce4, transparent); margin: 1.2rem 0; }

.widget-tab > .p-TabBar { background: #eef0f4; border-radius: 10px 10px 0 0; padding: 4px 6px 0; }
.widget-tab > .p-TabBar .p-TabBar-tab { font-family: 'Inter', sans-serif !important; font-size: 0.85rem !important; font-weight: 500; border-radius: 8px 8px 0 0; }
.widget-tab > .p-TabBar .p-TabBar-tab.p-mod-current { font-weight: 600 !important; background: white !important; }
.widget-tab > .widget-tab-contents { background: white; border-radius: 0 0 12px 12px; border: 1px solid #e8ecf0; padding: 1.2rem; box-shadow: 0 2px 8px rgba(0,0,0,0.05); }
</style>

In [ ]:
import pandas as pd
import numpy as np
import altair as alt
import plotly.express as px
import pycountry
import ipywidgets as widgets
from IPython.display import display, HTML
from vega_datasets import data as vega_data
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

alt.data_transformers.enable('vegafusion')

# ── OWID ──────────────────────────────────────────────────────────────────────
owid = pd.read_csv('owid-covid-data.csv')
owid['date'] = pd.to_datetime(owid['date'])

# ── Google Mobility ───────────────────────────────────────────────────────────
mob_raw = pd.read_csv('mobility_report_countries.csv')
mob_raw = mob_raw[mob_raw['region'] == 'Total']
mob_raw['date'] = pd.to_datetime(mob_raw['date'])
out_cols = ['retail and recreation', 'grocery and pharmacy', 'parks', 'transit stations', 'workplaces']
mob_raw['total_out'] = mob_raw[out_cols].mean(axis=1)
mob_raw['week'] = mob_raw['date'].dt.to_period('W')
wmobility_df = (
    mob_raw.groupby(['week', 'country'])
    .agg(total_out=('total_out', 'mean'), residential=('residential', 'mean'))
    .reset_index()
)

# ── OxCGRT ────────────────────────────────────────────────────────────────────
oxcgrt = pd.read_csv('OxCGRT_compact_national_v1.csv')
oxcgrt['Date'] = pd.to_datetime(oxcgrt['Date'], format='%Y%m%d')
oxcgrt['week'] = oxcgrt['Date'].dt.to_period('W')
weekly_stringency = (
    oxcgrt.groupby(['week', 'CountryName'])['StringencyIndex_Average']
    .mean().reset_index()
    .rename(columns={'CountryName': 'country'})
)

# ── ML feature data ───────────────────────────────────────────────────────────
selected_countries = [
    'United States', 'China', 'New Zealand', 'Sweden', 'Brazil', 'Germany',
    'United Kingdom', 'France', 'Italy', 'Spain', 'Canada', 'Australia',
    'Japan', 'South Korea', 'Singapore', 'Netherlands', 'Belgium', 'Denmark', 'Norway'
]
owid_ml = owid[owid['location'].isin(selected_countries)].copy()
owid_ml = owid_ml[~owid_ml['iso_code'].str.startswith('OWID')]
owid_ml = owid_ml.rename(columns={'location': 'country'})
owid_ml['week'] = owid_ml['date'].dt.to_period('W').dt.start_time
owid_ml = owid_ml.sort_values(['country', 'date'])

avg_cols  = [c for c in ['new_deaths_smoothed_per_million', 'new_cases_smoothed_per_million',
                          'stringency_index', 'reproduction_rate'] if c in owid_ml.columns]
last_cols = [c for c in ['gdp_per_capita', 'median_age', 'hospital_beds_per_thousand',
                          'population_density', 'aged_65_older', 'human_development_index',
                          'diabetes_prevalence', 'life_expectancy',
                          'people_vaccinated_per_hundred'] if c in owid_ml.columns]

agg_dict = {**{c: 'mean' for c in avg_cols}, **{c: 'last' for c in last_cols}}
weekly_ml = owid_ml.groupby(['country', 'week']).agg(agg_dict).reset_index()
weekly_ml = weekly_ml.sort_values(['country', 'week'])
weekly_ml[avg_cols] = weekly_ml.groupby('country')[avg_cols].ffill()
if 'people_vaccinated_per_hundred' in weekly_ml.columns:
    weekly_ml['people_vaccinated_per_hundred'] = weekly_ml['people_vaccinated_per_hundred'].fillna(0)

weekly_ml['stringency_lag_2'] = weekly_ml.groupby('country')['stringency_index'].shift(2)
weekly_ml = weekly_ml.dropna(subset=['new_deaths_smoothed_per_million'])

all_features = ['stringency_lag_2', 'new_cases_smoothed_per_million', 'aged_65_older',
                'human_development_index', 'population_density', 'reproduction_rate',
                'diabetes_prevalence', 'hospital_beds_per_thousand',
                'people_vaccinated_per_hundred', 'life_expectancy']
features = [f for f in all_features if f in weekly_ml.columns]
df_model = weekly_ml[features + ['new_deaths_smoothed_per_million', 'country', 'week']].dropna()

In [ ]:
countries_tracked = owid[
    owid['continent'].notna() & ~owid['iso_code'].str.startswith('OWID', na=True)
]['location'].nunique()

total_deaths = owid.groupby('location')['total_deaths'].max().sum()
peak_stringency = owid['stringency_index'].max()
date_range = f"{owid['date'].min().strftime('%b %Y')} \u2013 {owid['date'].max().strftime('%b %Y')}"

display(HTML(f'''
<div class="hero">
    <div class="hero-badge">\U0001f393 CS 329E \u00b7 University of Texas at Austin</div>
    <h1>\U0001f9a0 COVID-19 Policy &amp; Mortality Dashboard</h1>
    <p>Exploring how government responses shaped pandemic outcomes across 200+ countries &amp; territories</p>
</div>
<div class="kpi-row">
    <div class="kpi-card">
        <div class="kpi-icon">\U0001f30d</div>
        <div class="kpi-label">Countries Tracked</div>
        <div class="kpi-value">{countries_tracked}</div>
        <div class="kpi-sub">across 6 continents</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-icon">\U0001f4ca</div>
        <div class="kpi-label">Confirmed Deaths</div>
        <div class="kpi-value">{total_deaths/1e6:.1f}M</div>
        <div class="kpi-sub">cumulative worldwide</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-icon">\U0001f4cb</div>
        <div class="kpi-label">Peak Stringency</div>
        <div class="kpi-value">{peak_stringency:.0f}</div>
        <div class="kpi-sub">out of 100 (Oxford Index)</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-icon">\U0001f4c5</div>
        <div class="kpi-label">Date Range</div>
        <div class="kpi-value" style="font-size:1.2rem">{date_range}</div>
        <div class="kpi-sub">full pandemic timeline</div>
    </div>
</div>
<hr class="fancy-divider">
'''))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CHARTS 1–5
# ══════════════════════════════════════════════════════════════════════════════

# ── Chart 1: Deaths per million choropleth ────────────────────────────────────
def to_numeric(iso3):
    try:
        return int(pycountry.countries.get(alpha_3=iso3).numeric)
    except Exception:
        return None

deaths = owid.groupby(['iso_code', 'location'])['total_deaths_per_million'].max().reset_index()
deaths = deaths[deaths['total_deaths_per_million'] > 0]
deaths['id'] = deaths['iso_code'].apply(to_numeric)
deaths = deaths.dropna(subset=['id'])
deaths['id'] = deaths['id'].astype(int)

world = alt.topo_feature(vega_data.world_110m.url, 'countries')

chart1 = (
    alt.Chart(world).mark_geoshape(stroke='white', strokeWidth=0.4)
    .encode(
        color=alt.Color('total_deaths_per_million:Q',
                        scale=alt.Scale(scheme='redyellowgreen', reverse=True, type='log'),
                        legend=alt.Legend(title='Deaths / Million (log)', format='.0f', orient='bottom-right')),
        tooltip=[alt.Tooltip('location:N', title='Country'),
                 alt.Tooltip('total_deaths_per_million:Q', title='Deaths per Million', format=',.0f')],
    )
    .transform_lookup(lookup='id',
                      from_=alt.LookupData(data=deaths, key='id',
                                           fields=['total_deaths_per_million', 'location']))
    .project('naturalEarth1')
    .properties(width=900, height=480)
    .configure_view(strokeWidth=0)
)

# ── Chart 2: Stringency over time ─────────────────────────────────────────────
stringency_monthly = (
    owid[(owid['date'] >= '2020-01-01') & (owid['date'] <= '2022-12-31')
         & owid['stringency_index'].notna()]
    .groupby(['location', pd.Grouper(key='date', freq='MS')])['stringency_index']
    .mean().reset_index()
)

highlight2 = ['United States', 'China', 'New Zealand', 'Sweden', 'Brazil', 'Germany']
color_scale2 = alt.Scale(domain=highlight2,
                         range=['#1f77b4', '#d62728', '#2ca02c', '#8c564b', '#ff7f0e', '#9467bd'])

grey2 = (
    alt.Chart(stringency_monthly[~stringency_monthly['location'].isin(highlight2)])
    .mark_line(opacity=0.07, color='#aab4c4', strokeWidth=0.7)
    .encode(x=alt.X('date:T', title='',
                     axis=alt.Axis(format='%b %Y', tickCount={'interval': 'month', 'step': 3})),
            y=alt.Y('stringency_index:Q', title='Stringency Index', scale=alt.Scale(domain=[0, 100])),
            detail='location:N')
)

highlight_sel2 = alt.selection_point(fields=['location'], bind='legend')
lines2 = (
    alt.Chart(stringency_monthly[stringency_monthly['location'].isin(highlight2)])
    .mark_line(strokeWidth=2.5, interpolate='monotone')
    .encode(x='date:T', y='stringency_index:Q',
            color=alt.Color('location:N', scale=color_scale2, title='Country'),
            opacity=alt.condition(highlight_sel2, alt.value(1), alt.value(0.15)),
            tooltip=[alt.Tooltip('location:N', title='Country'),
                     alt.Tooltip('date:T', title='Date', format='%b %Y'),
                     alt.Tooltip('stringency_index:Q', title='Stringency Index', format='.1f')])
    .add_params(highlight_sel2)
)

events2 = pd.DataFrame([
    {'date': '2020-01-23', 'label': 'Wuhan Lockdown',      'y': 95},
    {'date': '2020-03-11', 'label': 'WHO Pandemic',        'y': 85},
    {'date': '2020-03-26', 'label': 'NZ Lockdown',         'y': 75},
    {'date': '2020-06-08', 'label': 'NZ Reopens',          'y': 95},
    {'date': '2021-01-20', 'label': 'US Mask Mandate',     'y': 85},
    {'date': '2021-07-01', 'label': 'Delta Wave',          'y': 75},
    {'date': '2021-12-01', 'label': 'Omicron Wave',        'y': 95},
    {'date': '2022-03-01', 'label': 'Restrictions Lifted', 'y': 85},
])
events2['date'] = pd.to_datetime(events2['date'])
ann_rules2 = alt.Chart(events2).mark_rule(strokeDash=[4, 4], opacity=0.35, color='#555').encode(x='date:T')
ann_text2  = (alt.Chart(events2)
              .mark_text(align='left', dx=5, dy=-5, fontSize=9, color='#555', fontStyle='italic')
              .encode(x='date:T', y='y:Q', text='label:N'))

waves2 = pd.DataFrame([{'start': '2021-06-01', 'end': '2021-10-01'},
                        {'start': '2021-11-15', 'end': '2022-02-15'}])
waves2['start'] = pd.to_datetime(waves2['start'])
waves2['end']   = pd.to_datetime(waves2['end'])
bands2 = alt.Chart(waves2).mark_rect(opacity=0.06, color='#0f3460').encode(x='start:T', x2='end:T')

chart2 = (
    (bands2 + grey2 + lines2 + ann_rules2 + ann_text2)
    .properties(width=900, height=460)
    .configure_axis(grid=True, gridOpacity=0.15, labelFontSize=11, titleFontSize=12)
    .configure_view(strokeWidth=0)
    .configure_legend(labelFontSize=11, titleFontSize=12)
)

# ── Chart 3: Policy timing vs mortality (brush + linked histogram) ────────────────────────────────
first_death3 = (owid[owid['total_deaths'] > 0].groupby('location')['date']
                .min().reset_index().rename(columns={'date': 'first_death_date'}))
strong_restr3 = (owid[owid['stringency_index'] >= 70].groupby('location')['date']
                 .min().reset_index().rename(columns={'date': 'restriction_date'}))
country_dates3 = pd.merge(first_death3, strong_restr3, on='location', how='inner')
country_dates3['days_to_restrictions'] = (
    country_dates3['restriction_date'] - country_dates3['first_death_date']).dt.days

deaths_pm3 = owid.groupby('location')['total_deaths_per_million'].max().reset_index()
meta3 = owid.groupby('location').agg(continent=('continent', 'first'),
                                     population=('population', 'first')).reset_index()
country_owid3 = country_dates3.merge(deaths_pm3, on='location').merge(meta3, on='location')
country_owid3 = country_owid3[country_owid3['continent'].notna()]
country_owid3 = country_owid3[country_owid3['days_to_restrictions'] < 200]
country_owid3['death_pct'] = (country_owid3['total_deaths_per_million'] / 1_000_000) * 100

label_countries3 = ['United States', 'Italy', 'China', 'India', 'Brazil',
                    'United Kingdom', 'Sweden', 'New Zealand']
continent_scale3 = alt.Scale(
    domain=['Africa', 'Asia', 'Europe', 'North America', 'South America', 'Oceania'])
brush3 = alt.selection_interval()

points3 = (
    alt.Chart(country_owid3).mark_circle(opacity=0.7, stroke='black', strokeWidth=0.3)
    .encode(
        x=alt.X('days_to_restrictions:Q',
                 title='Number of Days Between First COVID Death and Strong Restrictions',
                 scale=alt.Scale(domain=[-750, 150])),
        y=alt.Y('total_deaths_per_million:Q', title='COVID Deaths per Million'),
        color=alt.Color('continent:N', scale=continent_scale3),
        size=alt.Size('population:Q', title='Population', scale=alt.Scale(range=[30, 2000])),
        opacity=alt.condition(brush3, alt.value(0.7), alt.value(0.1)),
        tooltip=[alt.Tooltip('location:N', title='Country'),
                 alt.Tooltip('continent:N', title='Continent'),
                 alt.Tooltip('days_to_restrictions:Q', title='Days to Restrictions'),
                 alt.Tooltip('total_deaths_per_million:Q', title='Deaths per Million', format='.2f'),
                 alt.Tooltip('population:Q', title='Population', format=',')]
    )
    .add_params(brush3)
    .properties(width=800, height=500, title='Policy Timing vs COVID Deaths')
)
labels3 = (
    alt.Chart(country_owid3[country_owid3['location'].isin(label_countries3)])
    .mark_text(align='left', baseline='middle', dx=7, dy=-7, fontSize=11)
    .encode(x='days_to_restrictions:Q', y='total_deaths_per_million:Q',
            text='location:N', color=alt.value('#333333'))
)
zero_line3  = alt.Chart(pd.DataFrame({'x': [0]})).mark_rule(color='gray', strokeDash=[4, 4]).encode(x='x:Q')
zero_label3 = (
    alt.Chart(pd.DataFrame({'x': [-200], 'y': [6000],
                            'label': ['Restrictions implemented near first death']}))
    .mark_text(align='left', dx=7).encode(x='x:Q', y='y:Q', text='label:N')
)
hist3 = (
    alt.Chart(country_owid3).mark_bar()
    .encode(
        y=alt.Y('death_pct:Q', bin=alt.Bin(maxbins=30), title='COVID Deaths (% of Population)'),
        x=alt.X('count():Q', title='Number of Countries'),
        color=alt.Color('continent:N', scale=continent_scale3)
    )
    .transform_filter(brush3)
    .properties(width=500, height=200, title='COVID Deaths (% of Population) Distribution')
)
chart3 = (
    ((points3 + labels3 + zero_line3 + zero_label3) & hist3)
    .configure_axis(labelFontSize=12, titleFontSize=13)
    .configure_legend(titleFontSize=12, labelFontSize=11)
)

# ── Chart 4: Animated Plotly choropleth ───────────────────────────────────────
map_df4 = owid[owid['continent'].notna() & owid['iso_code'].notna()
               & owid['stringency_index'].notna()].copy()
map_df4 = map_df4[~map_df4['iso_code'].str.startswith('OWID')]
map_df4['month_label'] = map_df4['date'].dt.to_period('M').dt.to_timestamp().dt.strftime('%Y-%m')
monthly_s4 = map_df4.groupby(['iso_code', 'location', 'month_label'],
                              as_index=False)['stringency_index'].mean()

fig4 = px.choropleth(monthly_s4, locations='iso_code', color='stringency_index',
                     hover_name='location', animation_frame='month_label',
                     color_continuous_scale='OrRd', range_color=(0, 100),
                     projection='natural earth',
                     labels={'stringency_index': 'Stringency Index'})
fig4.update_layout(
    height=520, margin=dict(l=0, r=0, t=10, b=0),
    paper_bgcolor='rgba(0,0,0,0)',
    geo=dict(bgcolor='rgba(0,0,0,0)', showframe=False,
             showcoastlines=True, coastlinecolor='#ccc'),
    coloraxis_colorbar=dict(title='Stringency', thickness=14, len=0.6),
    font=dict(family='Inter, sans-serif'),
)

# ── Chart 5: Policy & mortality trends ────────────────────────────────────────
peer5 = ['United States', 'United Kingdom', 'Canada', 'Germany',
         'Italy', 'Sweden', 'Australia', 'Japan']
ts5 = (owid[owid['location'].isin(peer5)]
       [['date', 'location', 'stringency_index', 'new_deaths_per_million']].copy()
       .rename(columns={'new_deaths_per_million': 'mortality_rate'}))
ts5 = ts5[ts5['stringency_index'].notna() | ts5['mortality_rate'].notna()]
ts5['week'] = ts5['date'].dt.to_period('W').dt.start_time
weekly5 = (ts5.groupby(['week', 'location'], as_index=False)
           .agg(stringency_index=('stringency_index', 'mean'),
                mortality_rate=('mortality_rate', 'mean')))

sel5   = alt.selection_point(fields=['location'], bind='legend')
color5 = alt.Color('location:N', title='Country', scale=alt.Scale(scheme='tableau10'))

s_chart5 = (
    alt.Chart(weekly5).mark_line(size=2, interpolate='monotone')
    .encode(x=alt.X('week:T', title=''),
            y=alt.Y('stringency_index:Q', title='Stringency Index',
                    scale=alt.Scale(domain=[0, 100])),
            color=color5,
            opacity=alt.condition(sel5, alt.value(1), alt.value(0.1)),
            tooltip=[alt.Tooltip('week:T', title='Week'),
                     alt.Tooltip('location:N', title='Country'),
                     alt.Tooltip('stringency_index:Q', title='Stringency', format='.1f')])
    .add_params(sel5)
    .properties(width=900, height=250,
                title=alt.TitleParams('Policy Stringency Over Time',
                                      fontSize=13, fontWeight=600))
)
m_chart5 = (
    alt.Chart(weekly5).mark_line(size=2, interpolate='monotone')
    .encode(x=alt.X('week:T', title='Date'),
            y=alt.Y('mortality_rate:Q', title='New Deaths per Million'),
            color=color5,
            opacity=alt.condition(sel5, alt.value(1), alt.value(0.1)),
            tooltip=[alt.Tooltip('week:T', title='Week'),
                     alt.Tooltip('location:N', title='Country'),
                     alt.Tooltip('mortality_rate:Q', title='Deaths per Million', format='.2f')])
    .properties(width=900, height=250,
                title=alt.TitleParams('Mortality Trends Over Time',
                                      fontSize=13, fontWeight=600))
)
chart5 = (
    alt.vconcat(s_chart5, m_chart5).resolve_scale(color='shared')
    .configure_axis(grid=True, gridOpacity=0.12, labelFontSize=11, titleFontSize=12)
    .configure_view(strokeWidth=0)
    .configure_legend(titleFontSize=12, labelFontSize=11, orient='right')
)

# ══════════════════════════════════════════════════════════════════════════════
# CHARTS 6–9  (new)
# ══════════════════════════════════════════════════════════════════════════════

# ── Chart 6: Residential mobility over time ───────────────────────────────────
highlight_mob = ['United States', 'New Zealand', 'Sweden', 'Brazil', 'Germany']
color_mob = alt.Scale(domain=highlight_mob,
                      range=['#1f77b4', '#2ca02c', '#8c564b', '#ff7f0e', '#9467bd'])

plot_mob = wmobility_df.copy()
if str(plot_mob['week'].dtype).startswith('period'):
    plot_mob['week'] = plot_mob['week'].dt.start_time
else:
    plot_mob['week'] = pd.to_datetime(plot_mob['week'])

mob_grey6 = plot_mob[~plot_mob['country'].isin(highlight_mob)]
mob_hi6   = plot_mob[plot_mob['country'].isin(highlight_mob)]

grey_lines6 = (
    alt.Chart(mob_grey6)
    .transform_window(smoothed_residential='mean(residential)', frame=[-2, 2], groupby=['country'])
    .mark_line(opacity=0.35, color='lightgray', strokeWidth=0.8)
    .encode(x=alt.X('week:T', title='Week'),
            y=alt.Y('smoothed_residential:Q',
                    title='Residential Mobility (% Change from Baseline)',
                    scale=alt.Scale(domain=[-15, 80])),
            detail='country:N')
)
hi_lines6 = (
    alt.Chart(mob_hi6)
    .transform_window(smoothed_residential='mean(residential)', frame=[-2, 2], groupby=['country'])
    .mark_line(strokeWidth=2.5, interpolate='monotone')
    .encode(x='week:T',
            y=alt.Y('smoothed_residential:Q', scale=alt.Scale(domain=[-15, 80])),
            color=alt.Color('country:N', scale=color_mob, title='Country'),
            tooltip=[alt.Tooltip('week:T', title='Week'),
                     alt.Tooltip('country:N', title='Country'),
                     alt.Tooltip('smoothed_residential:Q', title='Residential Mobility', format='.2f')])
)

events6 = pd.DataFrame([
    {'date': '2020-01-23', 'label': 'Wuhan Lockdown',      'y': 75},
    {'date': '2020-03-11', 'label': 'WHO Pandemic',        'y': 68},
    {'date': '2020-03-26', 'label': 'NZ Lockdown',         'y': 56},
    {'date': '2020-06-08', 'label': 'NZ Reopens',          'y': 70},
    {'date': '2021-01-20', 'label': 'US Mask Mandate',     'y': 72},
    {'date': '2021-07-01', 'label': 'Delta Wave',          'y': 66},
    {'date': '2021-12-01', 'label': 'Omicron Wave',        'y': 68},
    {'date': '2022-03-01', 'label': 'Restrictions Lifted', 'y': 72},
])
events6['date'] = pd.to_datetime(events6['date'])
event_rules6 = (alt.Chart(events6)
                .mark_rule(strokeDash=[4, 4], size=1, opacity=0.4, color='#555').encode(x='date:T'))
event_text6  = (alt.Chart(events6)
                .mark_text(align='left', dx=5, dy=-5, fontSize=9, color='#555', fontStyle='italic')
                .encode(x='date:T', y='y:Q', text='label:N'))

chart6 = (
    (grey_lines6 + hi_lines6 + event_rules6 + event_text6)
    .properties(width=900, height=450,
                title=alt.TitleParams('Smoothed Residential Mobility Over Time Across Countries',
                                      fontSize=13, fontWeight=600))
    .configure_view(strokeWidth=0)
    .configure_axis(grid=True, gridOpacity=0.12, labelFontSize=11, titleFontSize=12)
    .configure_legend(labelFontSize=11, titleFontSize=12)
)

# ── Chart 7: Stringency vs residential mobility (per-country dropdown) ────────
merged7 = pd.merge(weekly_stringency, wmobility_df, on=['country', 'week'], how='inner')
if str(merged7['week'].dtype).startswith('period'):
    merged7['week'] = merged7['week'].dt.to_timestamp()
else:
    merged7['week'] = pd.to_datetime(merged7['week'])

plot7 = merged7.dropna(subset=['country', 'week', 'residential', 'StringencyIndex_Average']).copy()

corr7 = (plot7.groupby('country')
         .apply(lambda x: x['StringencyIndex_Average'].corr(x['residential']))
         .reset_index(name='corr_value'))
plot7 = plot7.merge(corr7, on='country', how='left')

country_list7   = sorted(plot7['country'].unique().tolist())
country_select7 = alt.param(
    name='SelectedCountry',
    bind=alt.binding_select(options=country_list7, name='Choose a country: '),
    value=country_list7[0]
)

base7 = alt.Chart(plot7).transform_filter('datum.country == SelectedCountry')

points7 = base7.mark_circle(size=70, opacity=0.6, color='#4c78a8').encode(
    x=alt.X('StringencyIndex_Average:Q', title='Stringency Index'),
    y=alt.Y('residential:Q', title='Residential Mobility (% Change from Baseline)'),
    tooltip=[alt.Tooltip('country:N', title='Country'),
             alt.Tooltip('week:T', title='Week'),
             alt.Tooltip('StringencyIndex_Average:Q', title='Stringency', format='.2f'),
             alt.Tooltip('residential:Q', title='Residential Mobility', format='.2f')]
)
trend7 = (base7.transform_regression('StringencyIndex_Average', 'residential')
          .mark_line(size=3, color='#e45756')
          .encode(x='StringencyIndex_Average:Q', y='residential:Q'))
r2_text7 = (
    base7.transform_regression('StringencyIndex_Average', 'residential', params=True)
    .transform_calculate(label="'R\u00b2 = ' + format(datum.rSquared, '.2f')")
    .mark_text(align='left', baseline='top', fontSize=13, fontWeight=600)
    .encode(x=alt.value(10), y=alt.value(10), text='label:N')
)
corr_text7 = (
    base7.transform_aggregate(corr_value='max(corr_value)')
    .transform_calculate(label="'Pearson r = ' + format(datum.corr_value, '.2f')")
    .mark_text(align='left', baseline='top', fontSize=13)
    .encode(x=alt.value(10), y=alt.value(30), text='label:N')
)

chart7 = (
    (points7 + trend7 + r2_text7 + corr_text7)
    .add_params(country_select7)
    .properties(width=700, height=420,
                title=alt.TitleParams('Residential Mobility vs Stringency Index by Country',
                                      fontSize=13, fontWeight=600))
    .configure_view(strokeWidth=0)
    .configure_axis(grid=True, gridOpacity=0.12, labelFontSize=11, titleFontSize=12)
)

# ── Charts 8 & 9: ML models ───────────────────────────────────────────────────
label_map = {
    'stringency_lag_2':                'Policy Stringency (2-wk lag)',
    'new_cases_smoothed_per_million':  'New Cases / Million',
    'aged_65_older':                   'Pop. Aged 65+',
    'human_development_index':         'Human Dev. Index',
    'population_density':              'Population Density',
    'reproduction_rate':               'Reproduction Rate',
    'diabetes_prevalence':             'Diabetes Prevalence',
    'hospital_beds_per_thousand':      'Hospital Beds / 1k',
    'people_vaccinated_per_hundred':   'Vaccination Rate',
    'life_expectancy':                 'Life Expectancy',
}

# Fit scaled OLS on full data for coefficient chart
scaler_all = StandardScaler()
X_all_sc   = scaler_all.fit_transform(df_model[features].values)
lr_all     = LinearRegression().fit(X_all_sc, df_model['new_deaths_smoothed_per_million'].values)

coef_df = pd.DataFrame({
    'Feature':     [label_map.get(f, f) for f in features],
    'Coefficient': lr_all.coef_,
}).sort_values('Coefficient')
coef_df['Direction'] = coef_df['Coefficient'].apply(
    lambda x: 'Increases Deaths' if x > 0 else 'Decreases Deaths')

chart8 = (
    alt.Chart(coef_df).mark_bar(size=18)
    .encode(
        x=alt.X('Coefficient:Q', title='Standardized Coefficient (OLS)'),
        y=alt.Y('Feature:N',
                sort=alt.EncodingSortField(field='Coefficient', order='ascending'),
                title=''),
        color=alt.Color('Direction:N',
                        scale=alt.Scale(domain=['Increases Deaths', 'Decreases Deaths'],
                                        range=['#e45756', '#4c78a8']),
                        legend=alt.Legend(title='Effect on Mortality', orient='bottom')),
        tooltip=[alt.Tooltip('Feature:N'), alt.Tooltip('Coefficient:Q', format='.3f')]
    )
    .properties(width=700, height=350,
                title=alt.TitleParams('What Predicts COVID Mortality? (Scaled OLS Coefficients)',
                                      fontSize=13, fontWeight=600))
    .configure_view(strokeWidth=0)
    .configure_axis(labelFontSize=11, titleFontSize=12)
    .configure_legend(labelFontSize=11, titleFontSize=12)
)

# Time-based split (80 / 20 by date — correct for time-series)
split_date9 = df_model['week'].quantile(0.8)
train9 = df_model[df_model['week'] <= split_date9]
test9  = df_model[df_model['week'] >  split_date9]

X_tr_raw = train9[features].values;  X_te_raw = test9[features].values
y_tr9    = train9['new_deaths_smoothed_per_million'].values
y_te9    = test9['new_deaths_smoothed_per_million'].values

scaler9  = StandardScaler()
X_tr_sc  = scaler9.fit_transform(X_tr_raw)
X_te_sc  = scaler9.transform(X_te_raw)

lr_simple9 = LinearRegression().fit(X_tr_sc[:, [0]], y_tr9)
lr_full9   = LinearRegression().fit(X_tr_sc, y_tr9)
rf9        = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr_raw, y_tr9)

r2_df9 = pd.DataFrame([
    {'Model': 'OLS: Stringency Only', 'r2': r2_score(y_te9, lr_simple9.predict(X_te_sc[:, [0]])), 'Type': 'Linear'},
    {'Model': 'OLS: Full Features',   'r2': r2_score(y_te9, lr_full9.predict(X_te_sc)),           'Type': 'Linear'},
    {'Model': 'Random Forest',        'r2': r2_score(y_te9, rf9.predict(X_te_raw)),               'Type': 'Tree'},
])

r2_chart9 = (
    alt.Chart(r2_df9).mark_bar(size=40)
    .encode(
        x=alt.X('Model:N', sort='-y', title='',
                 axis=alt.Axis(labelAngle=-15)),
        y=alt.Y('r2:Q', title='R² Score (held-out test set)'),
        color=alt.Color('Type:N',
                        scale=alt.Scale(domain=['Linear', 'Tree'],
                                        range=['#4c78a8', '#f28e2b']),
                        legend=alt.Legend(title='Model Type')),
        tooltip=[alt.Tooltip('Model:N'), alt.Tooltip('r2:Q', title='R²', format='.3f')]
    )
    .properties(width=750, height=280,
                title=alt.TitleParams('Model R² Comparison (Time-Based Split)',
                                      fontSize=13, fontWeight=600))
)

rf_imp_df = pd.DataFrame({
    'Feature':    [label_map.get(f, f) for f in features],
    'Importance': rf9.feature_importances_,
}).sort_values('Importance')

rf_imp_chart = (
    alt.Chart(rf_imp_df).mark_bar(size=18, color='#f28e2b')
    .encode(
        x=alt.X('Importance:Q', title='Feature Importance'),
        y=alt.Y('Feature:N',
                sort=alt.EncodingSortField(field='Importance', order='ascending'),
                title=''),
        tooltip=[alt.Tooltip('Feature:N'), alt.Tooltip('Importance:Q', format='.3f')]
    )
    .properties(width=750, height=300,
                title=alt.TitleParams('Random Forest Feature Importance',
                                      fontSize=13, fontWeight=600))
)

r2_zero = alt.Chart(pd.DataFrame({'y': [0]})).mark_rule(color='black', strokeWidth=1).encode(y='y:Q')
r2_labels = (
    alt.Chart(r2_df9).mark_text(dy=-8, fontSize=11, fontWeight=600)
    .encode(
        x=alt.X('Model:N', sort=alt.EncodingSortField(field='r2', order='descending')),
        y='r2:Q',
        text=alt.Text('r2:Q', format='.3f'),
        color=alt.condition('datum.r2 >= 0', alt.value('#1a1a2e'), alt.value('#e45756'))
    )
)
r2_chart9 = (r2_chart9 + r2_zero + r2_labels)

chart9 = (
    alt.vconcat(r2_chart9, rf_imp_chart)
    .configure_view(strokeWidth=0)
    .configure_axis(labelFontSize=11, titleFontSize=12)
    .configure_legend(labelFontSize=11, titleFontSize=12)
)

# ══════════════════════════════════════════════════════════════════════════════
# ASSEMBLE TAB WIDGET  (9 tabs)
# ══════════════════════════════════════════════════════════════════════════════

tabs_info = [
    ('🗺️  Deaths Map',
     'Cumulative COVID-19 deaths per million population (log scale). Sets the stage — where did the pandemic hit hardest?',
     chart1),
    ('📈  Stringency Over Time',
     'Monthly Oxford Stringency Index 2020–2022. Click a country in the legend to highlight it.',
     chart2),
    ('🎬  Animated Stringency Map',
     'Watch policy stringency evolve globally month by month. Press play or drag the slider.',
     fig4),
    ('📉  Policy & Mortality Trends',
     'Weekly stringency and mortality for 8 peer nations side by side. Click legend to isolate a country.',
     chart5),
    ('⏱️  Policy Timing vs Mortality',
     'Did acting faster before the first death reduce mortality? Drag to select a group — the histogram updates to show their death % distribution.',
     chart3),
    ('🚶  Residential Mobility',
     'Did policies actually change behavior? Smoothed residential mobility over time — grey = all countries, colored = key nations.',
     chart6),
    ('🔗  Stringency vs Mobility',
     'The direct relationship: stricter policies → more time at home? Use the dropdown to explore each country.',
     chart7),
    ('📊  OLS Coefficients',
     'Controlling for all factors simultaneously — which variables actually drive COVID mortality up or down?',
     chart8),
    ('🤖  Model Comparison',
     'Time-based 80/20 split — OLS vs Random Forest on held-out future data. Note: negative R² on OLS means the linear model fails to generalize to future time periods, while Random Forest adapts better.',
     chart9),
]
out_widgets = []
for title, desc, chart in tabs_info:
    out = widgets.Output()
    with out:
        display(HTML(f'<div class="section-header">{title.strip()}</div>'
                     f'<div class="section-desc">{desc}</div>'))
        if hasattr(chart, 'show'):
            chart.show()
        else:
            display(chart)
    out_widgets.append(out)

tab = widgets.Tab(children=out_widgets)
for i, (title, _, _) in enumerate(tabs_info):
    tab.set_title(i, title)

display(tab)